# Aula 01 — Fundamentos de Machine Learning

## Goal

Observar, em um experimento controlado, como a capacidade do modelo afeta o desempenho no treino e em dados separados. Ao final, você deverá explicar por que menor erro de treino não garante melhor generalização.

> Este notebook é o laboratório executável da Aula 01 do módulo de Machine Learning clássico.

## Setup

As bibliotecas abaixo já estão disponíveis no Google Colab. Em ambiente local, instale `numpy`, `pandas`, `matplotlib` e `scikit-learn`.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

print('Python:', sys.version.split()[0])
print('scikit-learn:', sklearn.__version__)

### Key Assumptions

- O dataset é sintético, balanceado e serve apenas para aprendizagem.
- A accuracy é usada porque as duas classes têm a mesma prevalência.
- O holdout é uma demonstração. A Aula 02 formalizará treino, validação e teste.
- Os limiares de diagnóstico são heurísticos; não provam generalização.

## Steps

### 1. Defina os parâmetros do experimento

Altere `NOISE`, `N_SAMPLES` e as profundidades, depois execute novamente todas as células.

In [ ]:
RANDOM_STATE = 42
N_SAMPLES = 800
NOISE = 0.28
VALIDATION_SIZE = 0.25
TREE_DEPTHS = [1, 3, None]

### 2. Gere e visualize os dados

In [ ]:
X, y = make_moons(
    n_samples=N_SAMPLES,
    noise=NOISE,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=VALIDATION_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', s=22, alpha=0.75)
ax.set(title='Dataset de treino: duas classes em forma de luas', xlabel='feature 1', ylabel='feature 2')
plt.show()

### 3. Treine modelos com capacidades diferentes

Os algoritmos são usados aqui como caixas-pretas didáticas. Eles serão estudados em aulas próprias.

In [ ]:
models = {
    'baseline': DummyClassifier(strategy='most_frequent'),
    'linear': make_pipeline(StandardScaler(), LogisticRegression()),
}
for depth in TREE_DEPTHS:
    suffix = 'sem_limite' if depth is None else f'depth_{depth}'
    models[f'arvore_{suffix}'] = DecisionTreeClassifier(
        max_depth=depth, random_state=RANDOM_STATE
    )

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    train_score = accuracy_score(y_train, model.predict(X_train))
    valid_score = accuracy_score(y_valid, model.predict(X_valid))
    rows.append({
        'modelo': name,
        'accuracy_treino': train_score,
        'accuracy_validacao': valid_score,
        'gap': train_score - valid_score,
    })

results = pd.DataFrame(rows).sort_values('accuracy_validacao', ascending=False)
results.round(3)

### 4. Compare as fronteiras de decisão

In [ ]:
def plot_boundary(ax, model, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 350),
        np.linspace(y_min, y_max, 350),
    )
    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
    ax.scatter(X_valid[:, 0], X_valid[:, 1], c=y_valid, cmap='coolwarm',
               s=20, edgecolor='white', linewidth=0.3)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, key, title in zip(
    axes,
    ['arvore_depth_1', 'arvore_depth_3', 'arvore_sem_limite'],
    ['Pouca flexibilidade', 'Complexidade útil', 'Flexibilidade excessiva'],
):
    plot_boundary(ax, models[key], title)

plt.tight_layout()
plt.show()

## Checks

As verificações abaixo não escolhem o melhor modelo. Elas confirmam que o experimento produziu os sinais didáticos esperados.

In [ ]:
by_name = results.set_index('modelo')
assert by_name.loc['baseline', 'accuracy_validacao'] == 0.5
assert by_name.loc['arvore_sem_limite', 'accuracy_treino'] == 1.0
assert by_name.loc['arvore_sem_limite', 'gap'] > 0.08
assert by_name.loc['arvore_depth_3', 'accuracy_validacao'] > by_name.loc['baseline', 'accuracy_validacao']
print('Checks concluídos: o exemplo exibe baseline, ajuste útil e sinal de overfitting.')

## Takeaways

Com a configuração padrão:

- o baseline obtém 0,500 de accuracy;
- a árvore com profundidade 3 captura a estrutura principal;
- a árvore ilimitada atinge 1,000 no treino, mas cai na validação;
- a diferença entre treino e validação é um sinal de investigação, não uma prova isolada.

### Desafios

1. Use `NOISE = 0.10` e depois `NOISE = 0.45`. O que muda?
2. Use `N_SAMPLES = 100` e depois `N_SAMPLES = 5_000`.
3. Adicione árvores com profundidade 2, 4 e 8.
4. Repita o experimento com dez seeds e calcule média e desvio-padrão.
5. Escreva o que os resultados sustentam e o que não sustentam.

## Next Steps

Siga para a Aula 02 — *Do problema ao experimento: features, target, splits e baseline*. Ela mostrará como transformar este exercício didático em um protocolo de avaliação defensável.